# Diffusion X-Ray Generator  

This notebook demonstrates a end-to-end diffusion workflow for generating chest X-ray images.  

It includes:  

- dataset loading  
- preprocessing  
- diffusion schedule setup  
- sinusoidal noise embeddings  
- U-Net denoiser definition  
- diffusion model training  
- image generation  
- diffusion-step comparison  

## 📖 Acknowledgements  

This implementation is adapted from diffusion model examples presented in *Generative Deep Learning 2nd Ed.* (David Foster), originally applied to natural image datasets (e.g., flowers).  

The code has been substantially modified and extended for medical imaging applications, including dataset handling, training configuration, and evaluation.   

If you build upon this work, please also consider citing the original source material where appropriate.

Adapted from example implementations provided in *Generative Deep Learning 2nd Ed.* (David Foster), with substantial modifications.
Reference Foster, D. (2023). Generative Deep Learning. 2nd Ed. O'Reilly.

In [ ]:
# imports
import math
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from tensorflow.keras import (
    activations,
    callbacks,
    layers,
    losses,
    metrics,
    models,
    optimizers,
    utils,
)

from keras.saving import register_keras_serializable

In [ ]:
# Plot style and reproducibility
plt.style.use("seaborn-v0_8-colorblind")
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Configuration
# Dataset / paths
DATASET_PATH = Path("data/chest_xray/train/NORMAL")
CHECKPOINT_DIR = Path("checkpoint")
OUTPUT_DIR = Path("outputs")
LOG_DIR = Path("logs")

CHECKPOINT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# Data / model settings
IMAGE_SIZE = 64
BATCH_SIZE = 64
DATASET_REPETITIONS = 5
LOAD_MODEL = False

NOISE_EMBEDDING_SIZE = 32
PLOT_DIFFUSION_STEPS = 20

# Training settings
EMA = 0.999
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100

## ⚠️ Dataset  

This notebook expects a local chest X-ray dataset folder.

The dataset is not included in this repository, but is publicly available on Kaggle under an MIT license:  

👉 https://www.kaggle.com/datasets/divyam6969/chest-xray-pneumonia-dataset

Example structure:  

data/ \
└── chest_xray/ \
└── train/ \
└──bacterial_pneumonia \
└──viral_pneumonia \
└──fungal_pneumonia

After downloading, place the dataset in the `data/` directory as shown above.  

However, the model should work with any X-ray dataset

In [ ]:
# sample one batch
def sample_batch(dataset):
    batch = dataset.take(1).get_single_element()
    return batch

In [ ]:
# display images
def display(images, n=10, size=(20, 3), cmap="gray_r", as_type="float32", save_to=None):
    if images is None:
        raise ValueError("No images provided to display().")

    images = np.array(images)
    if images.ndim == 3:
        images = np.expand_dims(images, axis=0)

    if as_type is not None:
        images = images.astype(as_type)

    plt.figure(figsize=size)

    for i in range(min(n, len(images))):
        plt.subplot(1, min(n, len(images)), i + 1)
        image = images[i]

        if image.shape[-1] == 1:
            image = image[..., 0]

        plt.imshow(image, cmap=cmap)
        plt.axis("off")

    plt.tight_layout()

    if save_to is not None:
        plt.savefig(save_to, bbox_inches="tight", dpi=200)

    plt.show()

In [ ]:
# load dataset
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset path not found: {DATASET_PATH}. "
        "Please place the chest X-ray images in the expected local folder."
    )

train_data = utils.image_dataset_from_directory(
    DATASET_PATH,
    labels=None,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=None,
    shuffle=True,
    seed=42,
    interpolation="bilinear",
)

In [ ]:
# preprocessing
def preprocess(img):
    img = tf.cast(img, "float32") / 255.0
    return img


train = train_data.map(lambda x: preprocess(x))
train = train.repeat(DATASET_REPETITIONS)
train = train.batch(BATCH_SIZE, drop_remainder=True)

In [ ]:
# display training sample
train_sample = sample_batch(train)
display(train_sample)

In [ ]:
# linear diffusion schedule
def linear_diffusion_schedule(diffusion_times):
    min_rate = 0.0001
    max_rate = 0.02

    betas = min_rate + diffusion_times * (max_rate - min_rate)
    alphas = 1.0 - betas

    alpha_bars = tf.math.cumprod(alphas, axis=0)
    signal_rates = tf.sqrt(alpha_bars)
    noise_rates = tf.sqrt(1.0 - alpha_bars)

    return noise_rates, signal_rates

In [ ]:
# cosine diffusion schedule
def cosine_diffusion_schedule(diffusion_times):
    signal_rates = tf.cos(diffusion_times * math.pi / 2)
    noise_rates = tf.sin(diffusion_times * math.pi / 2)
    return noise_rates, signal_rates

In [ ]:
# offset cosine training schedule
def offset_cosine_diffusion_schedule(diffusion_times):
    min_signal_rate = 0.02
    max_signal_rate = 0.95

    start_angle = tf.acos(max_signal_rate)
    end_angle = tf.acos(min_signal_rate)

    diffusion_angles = start_angle + diffusion_times * (end_angle - start_angle)

    signal_rates = tf.cos(diffusion_angles)
    noise_rates = tf.sin(diffusion_angles)

    return noise_rates, signal_rates

In [ ]:
# create schedule arrays
T = 1000
diffusion_times = tf.convert_to_tensor([x / T for x in range(T)], dtype=tf.float32)

linear_noise_rates, linear_signal_rates = linear_diffusion_schedule(diffusion_times)
cosine_noise_rates, cosine_signal_rates = cosine_diffusion_schedule(diffusion_times)
offset_noise_rates, offset_signal_rates = offset_cosine_diffusion_schedule(diffusion_times)

In [ ]:
# plot signal schedules
plt.figure(figsize=(8, 5))
plt.plot(diffusion_times, linear_signal_rates**2, linewidth=1.5, label="linear")
plt.plot(diffusion_times, cosine_signal_rates**2, linewidth=1.5, label="cosine")
plt.plot(diffusion_times, offset_signal_rates**2, linewidth=1.5, label="offset cosine")
plt.xlabel("Diffusion time")
plt.ylabel("Signal rate squared")
plt.title("Signal schedules")
plt.legend()
plt.show()

In [ ]:
# plot noise schedules
plt.figure(figsize=(8, 5))
plt.plot(diffusion_times, linear_noise_rates**2, linewidth=1.5, label="linear")
plt.plot(diffusion_times, cosine_noise_rates**2, linewidth=1.5, label="cosine")
plt.plot(diffusion_times, offset_noise_rates**2, linewidth=1.5, label="offset cosine")
plt.xlabel("Diffusion time")
plt.ylabel("Noise rate squared")
plt.title("Noise schedules")
plt.legend()
plt.show()

## Sinusoidal Noise Embedding  

Diffusion models require a representation of the current noise level (timestep).  
We use sinusoidal embeddings to encode this information and inject it into the U-Net.

In [ ]:
# sinusoidal embedding
@register_keras_serializable()
def sinusoidal_embedding(x):
    frequencies = tf.exp(
        tf.linspace(
            tf.math.log(1.0),
            tf.math.log(1000.0),
            NOISE_EMBEDDING_SIZE // 2,
        )
    )
    angular_speeds = 2.0 * math.pi * frequencies
    embeddings = tf.concat(
        [
            tf.sin(angular_speeds * x),
            tf.cos(angular_speeds * x),
        ],
        axis=3,
    )
    return embeddings

In [ ]:
# visualise embeddings
embedding_list = []
for y in np.arange(0, 1, 0.01):
    noise_variance = tf.ones((1, 1, 1, 1)) * y
    embedding = sinusoidal_embedding(noise_variance)
    embedding_list.append(embedding.numpy().reshape(-1))

embedding_array = np.array(embedding_list)

plt.figure(figsize=(10, 6))
plt.imshow(embedding_array.T, aspect="auto", origin="lower", cmap="viridis")
plt.xlabel("Diffusion step")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal embeddings")
plt.colorbar()
plt.show()

In [ ]:
# residual, up and down blocks
def ResidualBlock(width):
    def apply(x):
        input_width = x.shape[3]

        if input_width == width:
            residual = x
        else:
            residual = layers.Conv2D(width, kernel_size=1)(x)

        x = layers.BatchNormalization(center=False, scale=False)(x)
        x = layers.Conv2D(
            width,
            kernel_size=3,
            padding="same",
            activation=activations.swish,
        )(x)
        x = layers.Conv2D(width, kernel_size=3, padding="same")(x)
        x = layers.Add()([x, residual])

        return x

    return apply


def DownBlock(width, block_depth):
    def apply(x):
        x, skips = x
        for _ in range(block_depth):
            x = ResidualBlock(width)(x)
            skips.append(x)
        x = layers.AveragePooling2D(pool_size=2)(x)
        return x

    return apply


def UpBlock(width, block_depth):
    def apply(x):
        x, skips = x
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        for _ in range(block_depth):
            x = layers.Concatenate()([x, skips.pop()])
            x = ResidualBlock(width)(x)
        return x

    return apply

In [ ]:
# bluid U-net
noisy_images = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
x = layers.Conv2D(32, kernel_size=1)(noisy_images)

noise_variances = layers.Input(shape=(1, 1, 1))
noise_embedding = layers.Lambda(sinusoidal_embedding)(noise_variances)
noise_embedding = layers.UpSampling2D(size=IMAGE_SIZE, interpolation="nearest")(noise_embedding)

x = layers.Concatenate()([x, noise_embedding])

skips = []

x = DownBlock(32, block_depth=2)([x, skips])
x = DownBlock(64, block_depth=2)([x, skips])
x = DownBlock(96, block_depth=2)([x, skips])

x = ResidualBlock(128)(x)
x = ResidualBlock(128)(x)

x = UpBlock(96, block_depth=2)([x, skips])
x = UpBlock(64, block_depth=2)([x, skips])
x = UpBlock(32, block_depth=2)([x, skips])

x = layers.Conv2D(3, kernel_size=1, kernel_initializer="zeros")(x)

unet = models.Model([noisy_images, noise_variances], x, name="unet")

In [ ]:
# diffusion model class
class DiffusionModel(models.Model):
    def __init__(self):
        super().__init__()

        self.normalizer = layers.Normalization()
        self.network = unet
        self.ema_network = models.clone_model(self.network)
        self.diffusion_schedule = offset_cosine_diffusion_schedule

    def compile(self, **kwargs):
        super().compile(**kwargs)
        self.noise_loss_tracker = metrics.Mean(name="n_loss")

    @property
    def metrics(self):
        return [self.noise_loss_tracker]

    def denormalize(self, images):
        images = self.normalizer.mean + images * self.normalizer.variance**0.5
        return tf.clip_by_value(images, 0.0, 1.0)

    def denoise(self, noisy_images, noise_rates, signal_rates, training):
        network = self.network if training else self.ema_network
        pred_noises = network([noisy_images, noise_rates**2], training=training)
        pred_images = (noisy_images - noise_rates * pred_noises) / signal_rates
        return pred_noises, pred_images

    def reverse_diffusion(self, initial_noise, diffusion_steps):
        num_images = initial_noise.shape[0]
        step_size = 1.0 / diffusion_steps
        current_images = initial_noise

        for step in range(diffusion_steps):
            diffusion_times = tf.ones((num_images, 1, 1, 1)) - step * step_size
            noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)

            pred_noises, pred_images = self.denoise(
                current_images,
                noise_rates,
                signal_rates,
                training=False,
            )

            next_diffusion_times = diffusion_times - step_size
            next_noise_rates, next_signal_rates = self.diffusion_schedule(next_diffusion_times)

            current_images = (
                next_signal_rates * pred_images + next_noise_rates * pred_noises
            )

        return pred_images

    def generate(self, num_images, diffusion_steps, initial_noise=None):
        if initial_noise is None:
            initial_noise = tf.random.normal(
                shape=(num_images, IMAGE_SIZE, IMAGE_SIZE, 3)
            )

        generated_images = self.reverse_diffusion(initial_noise, diffusion_steps)
        generated_images = self.denormalize(generated_images)

        return generated_images

    def train_step(self, images):
        images = self.normalizer(images, training=True)
        noises = tf.random.normal(shape=tf.shape(images))

        diffusion_times = tf.random.uniform(
            shape=(tf.shape(images)[0], 1, 1, 1),
            minval=0.0,
            maxval=1.0,
        )

        noise_rates, signal_rates = self.diffusion_schedule(diffusion_times)
        noisy_images = signal_rates * images + noise_rates * noises

        with tf.GradientTape() as tape:
            pred_noises, _ = self.denoise(
                noisy_images,
                noise_rates,
                signal_rates,
                training=True,
            )
            noise_loss = self.compiled_loss(noises, pred_noises)

        gradients = tape.gradient(noise_loss, self.network.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.network.trainable_weights))

        self.noise_loss_tracker.update_state(noise_loss)

        for weight, ema_weight in zip(self.network.weights, self.ema_network.weights):
            ema_weight.assign(EMA * ema_weight + (1 - EMA) * weight)

        return {"n_loss": self.noise_loss_tracker.result()}

In [ ]:
# initialise model
ddm = DiffusionModel()
ddm.normalizer.adapt(train)

In [ ]:
# load weights
if LOAD_MODEL:
    ddm.built = True
    ddm.load_weights(CHECKPOINT_DIR / "checkpoint.weights.h5")

In [ ]:
# complie model
ddm.compile(
    optimizer=optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    ),
    loss=losses.MeanAbsoluteError(),
)

In [ ]:
# callbacks and training
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_DIR / "checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

tensorboard_callback = callbacks.TensorBoard(log_dir=LOG_DIR)


class ImageGenerator(callbacks.Callback):
    def __init__(self, num_img):
        self.num_img = num_img

    def on_epoch_end(self, epoch, logs=None):
        generated_images = self.model.generate(
            num_images=self.num_img,
            diffusion_steps=PLOT_DIFFUSION_STEPS,
        ).numpy()

        display(
            generated_images,
            save_to=OUTPUT_DIR / f"generated_xray_img_{epoch:03d}.png",
        )


image_generator_callback = ImageGenerator(num_img=10)

ddm.build(input_shape=(None, IMAGE_SIZE, IMAGE_SIZE, 3))

ddm.fit(
    train,
    epochs=EPOCHS,
    callbacks=[
        model_checkpoint_callback,
        tensorboard_callback,
        image_generator_callback,
    ],
)

In [ ]:
# generate sample images
generated_images = ddm.generate(num_images=10, diffusion_steps=20).numpy()
display(generated_images)

In [ ]:
# compare diffusion steps
for diffusion_steps in list(np.arange(1, 6, 1)) + [20] + [100]:
    tf.random.set_seed(42)
    generated_images = ddm.generate(
        num_images=10,
        diffusion_steps=diffusion_steps,
    ).numpy()

    display(
        generated_images,
        n=10,
        size=(20, 3),
        save_to=OUTPUT_DIR / f"xray_diffusion_steps_{diffusion_steps:03d}.png",
    )